In [60]:
import json
import os
import pandas as pd

In [62]:
DATA_PATH = "../data/raw/ipl_json"
json_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".json")]

len(json_files)   

1169

In [64]:
all_deliveries = [] 

In [66]:
# dismissal kinds that credit the bowler
BOWLER_KINDS = {"bowled", "caught", "caught and bowled", "lbw", "stumped", "hit wicket"}
# kinds that do NOT end the batter's innings as an "out" (for batting average)
NOT_OUT_KINDS = {"retired hurt", "retired not out"}

all_balls, all_wickets, all_matches, all_players = [], [], [], []

for file_name in json_files:
    with open(os.path.join(DATA_PATH, file_name), "r", encoding="utf-8") as f:
        m = json.load(f)

    info = m["info"]
    match_id = file_name.replace(".json", "")
    teams = info.get("teams", [])
    registry = info.get("registry", {}).get("people", {})   # name -> player id
    outcome = info.get("outcome", {})
    toss = info.get("toss", {})
    event = info.get("event", {})

    # ---------- match-level row ----------
    all_matches.append({
        "match_id": match_id,
        "date": info["dates"][0],
        "season_raw": info.get("season"),
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team1": teams[0] if len(teams) > 0 else None,
        "team2": teams[1] if len(teams) > 1 else None,
        "toss_winner": toss.get("winner"),
        "toss_decision": toss.get("decision"),
        "winner": outcome.get("winner"),
        "result": outcome.get("result"),                    # e.g. "no result", "tie"
        "win_by_runs": outcome.get("by", {}).get("runs"),
        "win_by_wickets": outcome.get("by", {}).get("wickets"),
        "method": outcome.get("method"),                    # D/L if applied
        "player_of_match": ", ".join(info.get("player_of_match", [])),
        "event_name": event.get("name"),
        "match_number": event.get("match_number"),
        "stage": event.get("stage"),                        # final, qualifier, etc.
    })

    # ---------- playing squads ----------
    for team, plist in info.get("players", {}).items():
        for name in plist:
            all_players.append({
                "match_id": match_id,
                "team": team,
                "player": name,
                "player_id": registry.get(name),
            })

    # ---------- innings (super overs skipped via flag) ----------
    innings_no = 0
    for inn in m.get("innings", []):
        if inn.get("super_over"):
            continue
        innings_no += 1
        batting_team = inn["team"]
        bowling_team = next((t for t in teams if t != batting_team), None)

        for over_data in inn.get("overs", []):
            over_no = over_data["over"]

            for ball_no, d in enumerate(over_data["deliveries"], start=1):
                runs = d["runs"]
                ex = d.get("extras", {})
                batter = d["batter"]
                bowler = d["bowler"]
                non_striker = d.get("non_striker")

                all_balls.append({
                    "match_id": match_id,
                    "innings": innings_no,
                    "batting_team": batting_team,
                    "bowling_team": bowling_team,
                    "over": over_no,                  # 0-indexed
                    "ball_in_over": ball_no,          # position in the over, includes wides/no-balls
                    "batter": batter,
                    "batter_id": registry.get(batter),
                    "bowler": bowler,
                    "bowler_id": registry.get(bowler),
                    "non_striker": non_striker,
                    "runs_batter": runs["batter"],
                    "runs_extras": runs["extras"],
                    "runs_total": runs["total"],
                    "wides": ex.get("wides", 0),
                    "noballs": ex.get("noballs", 0),
                    "byes": ex.get("byes", 0),
                    "legbyes": ex.get("legbyes", 0),
                    "penalty": ex.get("penalty", 0),
                    "is_wide": "wides" in ex,
                    "is_noball": "noballs" in ex,
                    "is_wicket": "wickets" in d,
                })

                # ---------- one row per dismissal ----------
                for w in d.get("wickets", []):
                    fielders = [fl.get("name") for fl in w.get("fielders", []) if fl.get("name")]
                    all_wickets.append({
                        "match_id": match_id,
                        "innings": innings_no,
                        "batting_team": batting_team,
                        "bowling_team": bowling_team,
                        "over": over_no,
                        "ball_in_over": ball_no,
                        "player_out": w["player_out"],
                        "player_out_id": registry.get(w["player_out"]),
                        "kind": w["kind"],
                        "bowler": bowler,
                        "bowler_id": registry.get(bowler),
                        "fielders": ", ".join(fielders),
                        "striker": batter,
                        "non_striker": non_striker,
                        "bowler_wicket": w["kind"] in BOWLER_KINDS,
                        "counts_as_out": w["kind"] not in NOT_OUT_KINDS,
                    })

# =========================
# BUILD DATAFRAMES
# =========================
df_ball = pd.DataFrame(all_balls)
df_wkts = pd.DataFrame(all_wickets)
df_match = pd.DataFrame(all_matches)
df_players = pd.DataFrame(all_players)

# derived columns on balls
df_ball["is_legal"] = ~(df_ball["is_wide"] | df_ball["is_noball"])   # counts for bowler's over
df_ball["batter_ball_faced"] = ~df_ball["is_wide"]                   # counts for batter
df_ball["bowler_runs"] = df_ball["runs_batter"] + df_ball["wides"] + df_ball["noballs"]

# clean dates, year and chronological order
df_match["date"] = pd.to_datetime(df_match["date"])
df_match["year"] = df_match["date"].dt.year
df_match = df_match.sort_values(["date", "match_id"]).reset_index(drop=True)
df_match["match_seq"] = df_match.index + 1       # 1 = oldest match

# =========================
# SANITY CHECKS
# =========================
print(df_ball.shape, df_wkts.shape, df_match.shape, df_players.shape)
print(df_match["date"].min(), "->", df_match["date"].max())
print(df_match["year"].value_counts().sort_index())

k_runs = df_ball.loc[df_ball["batter"] == "V Kohli", "runs_batter"].sum()
k_outs = ((df_wkts["player_out"] == "V Kohli") & df_wkts["counts_as_out"]).sum()
print("Kohli runs:", k_runs, "| outs:", k_outs)

# save so you never re-parse the JSON again (pip install pyarrow)
# ---- fix mixed-type columns before saving ----
df_match["season_raw"] = df_match["season_raw"].astype(str)
df_match["season"] = df_match["year"]
df_match["match_number"] = pd.to_numeric(df_match["match_number"], errors="coerce")

# ---- save so you never re-parse the JSON again ----
df_ball.to_parquet("balls.parquet", index=False)
df_wkts.to_parquet("wickets.parquet", index=False)
df_match.to_parquet("matches.parquet", index=False)
df_players.to_parquet("players.parquet", index=False)
print("saved all 4 files")

(278034, 25) (13794, 16) (1169, 20) (26137, 4)
2008-04-18 00:00:00 -> 2025-06-03 00:00:00
year
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74
Name: count, dtype: int64
Kohli runs: 8661 | outs: 219
saved all 4 files


In [67]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

for name, df in [("df_ball", df_ball), ("df_wkts", df_wkts),
                 ("df_match", df_match), ("df_players", df_players)]:
    print(f"\n===== {name} {df.shape} =====")
    display(df.head())


===== df_ball (278034, 25) =====


,match_id,innings,batting_team,bowling_team,over,ball_in_over,batter,batter_id,bowler,bowler_id,non_striker,runs_batter,runs_extras,runs_total,wides,noballs,byes,legbyes,penalty,is_wide,is_noball,is_wicket,is_legal,batter_ball_faced,bowler_runs
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,1,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,2,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,3,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,4,0,4,0,0,0,0,0,False,False,False,True,True,4
3,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,4,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0
4,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,5,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,2,2,2,0,0,0,0,True,False,False,False,False,2



===== df_wkts (13794, 16) =====


,match_id,innings,batting_team,bowling_team,over,ball_in_over,player_out,player_out_id,kind,bowler,bowler_id,fielders,striker,non_striker,bowler_wicket,counts_as_out
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,dcce6f09,caught,A Choudhary,18e6906e,Mandeep Singh,DA Warner,S Dhawan,True,True
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,10,3,S Dhawan,0a476045,caught,STR Binny,bd17b45f,Sachin Baby,S Dhawan,MC Henriques,True,True
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,15,2,MC Henriques,32198ae0,caught,YS Chahal,57ee1fde,Sachin Baby,MC Henriques,Yuvraj Singh,True,True
3,1082591,1,Sunrisers Hyderabad,Royal Challengers Bangalore,18,4,Yuvraj Singh,1c914163,bowled,TS Mills,245c97cb,,Yuvraj Singh,DJ Hooda,True,True
4,1082591,2,Royal Challengers Bangalore,Sunrisers Hyderabad,5,4,Mandeep Singh,c3a96caf,bowled,Rashid Khan,5f547c8b,,Mandeep Singh,CH Gayle,True,True



===== df_match (1169, 21) =====


,match_id,date,season_raw,venue,city,team1,team2,toss_winner,toss_decision,winner,result,win_by_runs,win_by_wickets,method,player_of_match,event_name,match_number,stage,year,match_seq,season
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,None,140.0,NaN,None,BB McCullum,Indian Premier League,1.0,None,2008,1,2008
1,335983,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,None,33.0,NaN,None,MEK Hussey,Indian Premier League,2.0,None,2008,2,2008
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,None,NaN,9.0,None,MF Maharoof,Indian Premier League,3.0,None,2008,3,2008
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,None,NaN,5.0,None,MV Boucher,Indian Premier League,5.0,None,2008,4,2008
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,None,NaN,5.0,None,DJ Hussey,Indian Premier League,4.0,None,2008,5,2008



===== df_players (26137, 4) =====


,match_id,team,player,player_id
0,1082591,Royal Challengers Bangalore,CH Gayle,db584dad
1,1082591,Royal Challengers Bangalore,Mandeep Singh,c3a96caf
2,1082591,Royal Challengers Bangalore,TM Head,12b610c2
3,1082591,Royal Challengers Bangalore,KM Jadhav,99d63244
4,1082591,Royal Challengers Bangalore,SR Watson,4329fbb5


In [68]:
TEAM_ALIASES = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

for col in ["team1", "team2", "toss_winner", "winner"]:
    df_match[col] = df_match[col].replace(TEAM_ALIASES)
for col in ["batting_team", "bowling_team"]:
    df_ball[col] = df_ball[col].replace(TEAM_ALIASES)
    df_wkts[col] = df_wkts[col].replace(TEAM_ALIASES)
df_players["team"] = df_players["team"].replace(TEAM_ALIASES)

df_match = df_match.sort_values(["date", "match_number", "match_id"]).reset_index(drop=True)
df_match["match_seq"] = df_match.index + 1

# re-save the cleaned versions
df_ball.to_parquet("balls.parquet", index=False)
df_wkts.to_parquet("wickets.parquet", index=False)
df_match.to_parquet("matches.parquet", index=False)
df_players.to_parquet("players.parquet", index=False)

# --- checks: paste this output to me ---
print(sorted(df_match["team1"].unique()))
print(df_match["venue"].value_counts().to_string())
print(df_match[df_match["winner"].isna()][["match_id", "date", "team1", "team2", "result"]])

['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
venue
Eden Gardens                                                               77
Wankhede Stadium                                                           73
M Chinnaswamy Stadium                                                      65
Feroz Shah Kotla                                                           60
Wankhede Stadium, Mumbai                                                   52
Rajiv Gandhi International Stadium, Uppal                                  49
MA Chidambaram Stadium, Chepauk                                            48
Sawai Mansingh Stadium                                                     47
Dubai International Cricket Stadium               

In [72]:
VENUES = {
    "Eden Gardens, Kolkata": ["Eden Gardens", "Eden Gardens, Kolkata"],
    "Wankhede Stadium, Mumbai": ["Wankhede Stadium", "Wankhede Stadium, Mumbai"],
    "M Chinnaswamy Stadium, Bengaluru": ["M Chinnaswamy Stadium", "M Chinnaswamy Stadium, Bengaluru", "M.Chinnaswamy Stadium"],
    "Arun Jaitley Stadium, Delhi": ["Feroz Shah Kotla", "Arun Jaitley Stadium", "Arun Jaitley Stadium, Delhi"],
    "Rajiv Gandhi International Stadium, Hyderabad": ["Rajiv Gandhi International Stadium, Uppal", "Rajiv Gandhi International Stadium, Uppal, Hyderabad", "Rajiv Gandhi International Stadium"],
    "MA Chidambaram Stadium, Chennai": ["MA Chidambaram Stadium, Chepauk", "MA Chidambaram Stadium, Chepauk, Chennai", "MA Chidambaram Stadium"],
    "Sawai Mansingh Stadium, Jaipur": ["Sawai Mansingh Stadium", "Sawai Mansingh Stadium, Jaipur"],
    "PCA IS Bindra Stadium, Mohali": ["Punjab Cricket Association Stadium, Mohali", "Punjab Cricket Association IS Bindra Stadium, Mohali", "Punjab Cricket Association IS Bindra Stadium", "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh"],
    "Narendra Modi Stadium, Ahmedabad": ["Narendra Modi Stadium, Ahmedabad", "Sardar Patel Stadium, Motera"],
    "MCA Stadium, Pune": ["Maharashtra Cricket Association Stadium", "Maharashtra Cricket Association Stadium, Pune", "Subrata Roy Sahara Stadium"],
    "Dr DY Patil Sports Academy, Navi Mumbai": ["Dr DY Patil Sports Academy", "Dr DY Patil Sports Academy, Mumbai"],
    "Brabourne Stadium, Mumbai": ["Brabourne Stadium", "Brabourne Stadium, Mumbai"],
    "Dubai International Cricket Stadium, Dubai": ["Dubai International Cricket Stadium"],
    "Sheikh Zayed Stadium, Abu Dhabi": ["Sheikh Zayed Stadium", "Zayed Cricket Stadium, Abu Dhabi"],
    "Sharjah Cricket Stadium, Sharjah": ["Sharjah Cricket Stadium"],
    "Ekana Cricket Stadium, Lucknow": ["Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow"],
    "ACA-VDCA Stadium, Visakhapatnam": ["Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium", "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam"],
    "HPCA Stadium, Dharamsala": ["Himachal Pradesh Cricket Association Stadium", "Himachal Pradesh Cricket Association Stadium, Dharamsala"],
    "Maharaja Yadavindra Singh Stadium, Mullanpur": ["Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur", "Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh"],
    # single-spelling venues
    "Kingsmead, Durban": ["Kingsmead"],
    "SuperSport Park, Centurion": ["SuperSport Park"],
    "New Wanderers Stadium, Johannesburg": ["New Wanderers Stadium"],
    "St George's Park, Gqeberha": ["St George's Park"],
    "Newlands, Cape Town": ["Newlands"],
    "De Beers Diamond Oval, Kimberley": ["De Beers Diamond Oval"],
    "Buffalo Park, East London": ["Buffalo Park"],
    "OUTsurance Oval, Bloemfontein": ["OUTsurance Oval"],
    "Saurashtra Cricket Association Stadium, Rajkot": ["Saurashtra Cricket Association Stadium"],
    "Holkar Cricket Stadium, Indore": ["Holkar Cricket Stadium"],
    "JSCA International Stadium Complex, Ranchi": ["JSCA International Stadium Complex"],
    "Barabati Stadium, Cuttack": ["Barabati Stadium"],
    "Shaheed Veer Narayan Singh Stadium, Raipur": ["Shaheed Veer Narayan Singh International Stadium"],
    "Nehru Stadium, Kochi": ["Nehru Stadium"],
    "Barsapara Cricket Stadium, Guwahati": ["Barsapara Cricket Stadium, Guwahati"],
    "Green Park, Kanpur": ["Green Park"],
    "Vidarbha Cricket Association Stadium, Nagpur": ["Vidarbha Cricket Association Stadium, Jamtha"],
}
VENUE_MAP = {v: canon for canon, variants in VENUES.items() for v in variants}

df_match["venue_raw"] = df_match["venue"]
df_match["venue"] = df_match["venue_raw"].map(VENUE_MAP)
df_match["city"] = df_match["venue"].str.split(", ").str[-1]

print("unmapped:", df_match.loc[df_match["venue"].isna(), "venue_raw"].unique())
print(df_match["venue"].nunique(), "venues (expected 36)")
print(df_match["venue"].value_counts().head(10))

df_match.to_parquet("matches.parquet", index=False)

unmapped: []
36 venues (expected 36)
venue
Wankhede Stadium, Mumbai                         125
Eden Gardens, Kolkata                            100
M Chinnaswamy Stadium, Bengaluru                  99
Arun Jaitley Stadium, Delhi                       97
MA Chidambaram Stadium, Chennai                   91
Rajiv Gandhi International Stadium, Hyderabad     83
Sawai Mansingh Stadium, Jaipur                    64
PCA IS Bindra Stadium, Mohali                     61
MCA Stadium, Pune                                 51
Dubai International Cricket Stadium, Dubai        46
Name: count, dtype: int64


In [74]:
import os, json
tie_ids = df_match.loc[df_match["result"] == "tie", "match_id"]
for mid in tie_ids:
    with open(os.path.join(DATA_PATH, f"{mid}.json"), encoding="utf-8") as f:
        print(mid, json.load(f)["info"]["outcome"])

392190 {'eliminator': 'Rajasthan Royals', 'result': 'tie'}
419121 {'eliminator': 'Kings XI Punjab', 'result': 'tie'}
598004 {'eliminator': 'Sunrisers Hyderabad', 'result': 'tie'}
598017 {'eliminator': 'Royal Challengers Bangalore', 'result': 'tie'}
729315 {'eliminator': 'Rajasthan Royals', 'result': 'tie'}
829741 {'eliminator': 'Kings XI Punjab', 'result': 'tie'}
1082625 {'eliminator': 'Mumbai Indians', 'result': 'tie'}
1175365 {'result': 'tie', 'eliminator': 'Delhi Capitals'}
1178426 {'result': 'tie', 'eliminator': 'Mumbai Indians'}
1216493 {'result': 'tie', 'eliminator': 'Delhi Capitals'}
1216547 {'result': 'tie', 'eliminator': 'Royal Challengers Bangalore'}
1216512 {'result': 'tie', 'eliminator': 'Kolkata Knight Riders'}
1216517 {'result': 'tie', 'eliminator': 'Kings XI Punjab'}
1254077 {'result': 'tie', 'eliminator': 'Delhi Capitals'}
1473469 {'result': 'tie', 'eliminator': 'Delhi Capitals'}


In [76]:
import os, json

super_win = {}
for mid in df_match.loc[df_match["result"] == "tie", "match_id"]:
    with open(os.path.join(DATA_PATH, f"{mid}.json"), encoding="utf-8") as f:
        super_win[mid] = json.load(f)["info"]["outcome"].get("eliminator")

df_match["super_over_winner"] = df_match["match_id"].map(super_win).replace(TEAM_ALIASES)
df_match["final_winner"] = df_match["winner"].fillna(df_match["super_over_winner"])
df_match["is_no_result"] = df_match["result"].eq("no result")

print(df_match["final_winner"].isna().sum(), "without a winner (expect 8 no-results)")
df_match.to_parquet("matches.parquet", index=False)

8 without a winner (expect 8 no-results)


In [78]:
# ---------- bowling per bowler per innings ----------
b = df_ball.copy()
b["legal"] = b["is_legal"].astype(int)
b["dot"] = (b["is_legal"] & (b["runs_batter"] == 0)).astype(int)   # byes/leg-byes still count as dots
b["four_c"] = (b["runs_batter"] == 4).astype(int)
b["six_c"] = (b["runs_batter"] == 6).astype(int)

bowl = (
    b.groupby(["match_id", "innings", "bowling_team", "batting_team", "bowler"])
     .agg(balls=("legal", "sum"),
          runs=("bowler_runs", "sum"),
          dots=("dot", "sum"),
          wides=("wides", "sum"),
          noballs=("noballs", "sum"),
          fours=("four_c", "sum"),
          sixes=("six_c", "sum"))
     .reset_index()
     .rename(columns={"bowling_team": "team", "batting_team": "opposition"})
)

wk = (
    df_wkts[df_wkts["bowler_wicket"]]
    .groupby(["match_id", "innings", "bowler"]).size().rename("wickets").reset_index()
)
bowl = bowl.merge(wk, on=["match_id", "innings", "bowler"], how="left")
bowl["wickets"] = bowl["wickets"].fillna(0).astype(int)

bowl = bowl.merge(df_match[["match_id", "date", "year", "venue", "match_seq"]], on="match_id", how="left")
bowl["bowler_id"] = bowl["bowler"].map(df_players.drop_duplicates("player").set_index("player")["player_id"])
bowl["economy"] = bowl["runs"] / (bowl["balls"].where(bowl["balls"] > 0) / 6)
bowl = bowl.sort_values(["bowler", "match_seq", "innings"]).reset_index(drop=True)
bowl["inning_no"] = bowl.groupby("bowler").cumcount() + 1
bowl.to_parquet("player_match_bowling.parquet", index=False)

# ---------- bowling per season ----------
bseason = (
    bowl.groupby(["bowler", "year"])
        .agg(team=("team", lambda s: s.mode().iat[0]),
             innings=("match_id", "count"),
             balls=("balls", "sum"),
             runs=("runs", "sum"),
             wickets=("wickets", "sum"),
             dots=("dots", "sum"),
             best_wkts=("wickets", "max"),
             three_w=("wickets", lambda s: (s >= 3).sum()),
             five_w=("wickets", lambda s: (s >= 5).sum()))
        .reset_index()
)
bseason["economy"] = bseason["runs"] / (bseason["balls"] / 6)
bseason["avg"] = bseason["runs"] / bseason["wickets"].where(bseason["wickets"] > 0)
bseason["sr"] = bseason["balls"] / bseason["wickets"].where(bseason["wickets"] > 0)
bseason.to_parquet("player_season_bowling.parquet", index=False)

# ---------- checks ----------
for p in ["DJ Bravo", "SL Malinga", "YS Chahal"]:
    x = bowl[bowl["bowler"] == p]
    print(p, "| innings:", len(x), "| wickets:", x["wickets"].sum(),
          "| economy:", round(x["runs"].sum() / (x["balls"].sum() / 6), 2))

DJ Bravo | innings: 158 | wickets: 183 | economy: 8.38
SL Malinga | innings: 122 | wickets: 170 | economy: 7.14
YS Chahal | innings: 172 | wickets: 221 | economy: 7.96


In [80]:
# ---------- phase labels (over is 0-indexed) ----------
def to_phase(o):
    return "powerplay" if o <= 5 else ("middle" if o <= 14 else "death")

ymap = df_match.set_index("match_id")["year"]
for d in (df_ball, df_wkts):
    d["phase"] = d["over"].map(to_phase)
    d["year"] = d["match_id"].map(ymap)

df_ball["is_four"] = (df_ball["runs_batter"] == 4).astype(int)
df_ball["is_six"] = (df_ball["runs_batter"] == 6).astype(int)
df_ball["bat_dot"] = (df_ball["batter_ball_faced"] & (df_ball["runs_batter"] == 0)).astype(int)

# ---------- batting by phase ----------
bat_phase = (df_ball.groupby(["batter", "year", "phase"])
    .agg(runs=("runs_batter", "sum"), balls=("batter_ball_faced", "sum"),
         fours=("is_four", "sum"), sixes=("is_six", "sum"), dots=("bat_dot", "sum"))
    .reset_index())
outs_ph = (df_wkts[df_wkts["counts_as_out"]]
    .groupby(["player_out", "year", "phase"]).size().rename("outs").reset_index()
    .rename(columns={"player_out": "batter"}))
bat_phase = bat_phase.merge(outs_ph, on=["batter", "year", "phase"], how="outer").fillna(0)
for c in ["runs", "balls", "fours", "sixes", "dots", "outs"]:
    bat_phase[c] = bat_phase[c].astype(int)
bat_phase["sr"] = bat_phase["runs"] / bat_phase["balls"].where(bat_phase["balls"] > 0) * 100
bat_phase["avg"] = bat_phase["runs"] / bat_phase["outs"].where(bat_phase["outs"] > 0)

# ---------- bowling by phase ----------
bowl_phase = (df_ball.groupby(["bowler", "year", "phase"])
    .agg(balls=("is_legal", "sum"), runs=("bowler_runs", "sum")).reset_index())
wk_ph = (df_wkts[df_wkts["bowler_wicket"]]
    .groupby(["bowler", "year", "phase"]).size().rename("wickets").reset_index())
bowl_phase = bowl_phase.merge(wk_ph, on=["bowler", "year", "phase"], how="left")
bowl_phase["wickets"] = bowl_phase["wickets"].fillna(0).astype(int)
bowl_phase["economy"] = bowl_phase["runs"] / (bowl_phase["balls"].where(bowl_phase["balls"] > 0) / 6)

# ---------- batter vs bowler ----------
h2h = (df_ball.groupby(["batter", "bowler"])
    .agg(runs=("runs_batter", "sum"), balls=("batter_ball_faced", "sum"),
         dots=("bat_dot", "sum"), fours=("is_four", "sum"), sixes=("is_six", "sum"))
    .reset_index())
h_out = (df_wkts[df_wkts["bowler_wicket"]]
    .groupby(["player_out", "bowler"]).size().rename("outs").reset_index()
    .rename(columns={"player_out": "batter"}))
h2h = h2h.merge(h_out, on=["batter", "bowler"], how="left")
h2h["outs"] = h2h["outs"].fillna(0).astype(int)
h2h["sr"] = h2h["runs"] / h2h["balls"].where(h2h["balls"] > 0) * 100

# ---------- batter vs team, batter at venue (from the player-match table) ----------
def split(df, key):
    t = (df.groupby(["batter", key])
           .agg(innings=("match_id", "count"), runs=("runs", "sum"), balls=("balls", "sum"),
                outs=("outs", "sum"), high_score=("runs", "max")).reset_index())
    t["avg"] = t["runs"] / t["outs"].where(t["outs"] > 0)
    t["sr"] = t["runs"] / t["balls"].where(t["balls"] > 0) * 100
    return t

vs_team = split(bat, "opposition")
at_venue = split(bat, "venue")

# ---------- save ----------
bat_phase.to_parquet("batting_by_phase.parquet", index=False)
bowl_phase.to_parquet("bowling_by_phase.parquet", index=False)
h2h.to_parquet("batter_vs_bowler.parquet", index=False)
vs_team.to_parquet("batter_vs_team.parquet", index=False)
at_venue.to_parquet("batter_at_venue.parquet", index=False)

# ---------- checks ----------
k = bat_phase[bat_phase["batter"] == "V Kohli"]
print(k.groupby("phase")[["runs", "balls", "outs"]].sum())
print("totals:", k["runs"].sum(), k["balls"].sum(), "(expect 8661 runs, 6519 balls)")
print(h2h[(h2h["balls"] >= 30)].sort_values("runs", ascending=False).head(5))

           runs  balls  outs
phase                       
death      1507    808    60
middle     4037   3233    90
powerplay  3117   2478    69
totals: 8661 6519 (expect 8661 runs, 6519 balls)
          batter     bowler  runs  balls  dots  fours  sixes  outs          sr
5820   DA Warner  SP Narine   195    123    45     20      9     2  158.536585
27724    V Kohli   R Ashwin   181    149    44     13      6     1  121.476510
27733    V Kohli  RA Jadeja   179    160    51     10      5     3  111.875000
23803   SK Raina  PP Chawla   175    101    29     15     10     4  173.267327
20940  RG Sharma   UT Yadav   170    118    54     21      8     5  144.067797


In [82]:
df_ball.to_parquet("balls.parquet", index=False)
df_wkts.to_parquet("wickets.parquet", index=False)

In [84]:
import glob
for f in sorted(glob.glob("*.parquet")):
    print(f)

balls.parquet
batter_at_venue.parquet
batter_vs_bowler.parquet
batter_vs_team.parquet
batting_by_phase.parquet
bowling_by_phase.parquet
career_batting.parquet
career_bowling.parquet
matches.parquet
player_match_batting.parquet
player_match_bowling.parquet
player_season_batting.parquet
player_season_bowling.parquet
players.parquet
wickets.parquet


In [86]:
# ---------- season-wise batting ----------
season = (
    bat.groupby(["batter", "year"])
    .agg(team=("batting_team", lambda s: s.mode().iat[0]),
         innings=("match_id", "count"),
         runs=("runs", "sum"),
         balls=("balls", "sum"),
         outs=("outs", "sum"),
         fours=("fours", "sum"),
         sixes=("sixes", "sum"),
         high_score=("runs", "max"),
         fifties=("runs", lambda s: ((s >= 50) & (s < 100)).sum()),
         hundreds=("runs", lambda s: (s >= 100).sum()))
    .reset_index()
)
season["avg"] = season["runs"] / season["outs"].where(season["outs"] > 0)
season["sr"] = season["runs"] / season["balls"] * 100
season["runs_per_inn"] = season["runs"] / season["innings"]
season.to_parquet("player_season_batting.parquet", index=False)

# ---------- rolling form (last 10 innings) ----------
W = 10
g = bat.groupby("batter")
for col in ["runs", "balls", "outs"]:
    bat[f"roll_{col}"] = g[col].transform(lambda s: s.rolling(W, min_periods=5).sum())
bat["roll_sr"] = bat["roll_runs"] / bat["roll_balls"] * 100
bat["roll_avg"] = bat["roll_runs"] / bat["roll_outs"].where(bat["roll_outs"] > 0)
bat.to_parquet("player_match_batting.parquet", index=False)

# ---------- peaks and slumps ----------
def career_events(player, min_inn=6):
    s = season[(season["batter"] == player) & (season["innings"] >= min_inn)].copy()
    if len(s) < 3:
        return None
    mu, sd = s["runs_per_inn"].mean(), s["runs_per_inn"].std()
    s["z"] = (s["runs_per_inn"] - mu) / sd if sd > 0 else 0
    s["yoy_change"] = s["runs_per_inn"].diff()
    return {
        "best_season":  s.loc[s["runs_per_inn"].idxmax(), ["year", "team", "runs", "avg", "sr"]].to_dict(),
        "worst_season": s.loc[s["runs_per_inn"].idxmin(), ["year", "team", "runs", "avg", "sr"]].to_dict(),
        "biggest_jump": s.loc[s["yoy_change"].idxmax(), ["year", "yoy_change"]].to_dict(),
        "biggest_drop": s.loc[s["yoy_change"].idxmin(), ["year", "yoy_change"]].to_dict(),
        "peak_years":   s.loc[s["z"] >= 1, "year"].tolist(),
        "slump_years":  s.loc[s["z"] <= -1, "year"].tolist(),
    }

print(season[season["batter"] == "V Kohli"].round(2).to_string(index=False))
print(career_events("V Kohli"))

 batter  year                        team  innings  runs  balls  outs  fours  sixes  high_score  fifties  hundreds   avg     sr  runs_per_inn
V Kohli  2008 Royal Challengers Bangalore       12   165    157    11     18      4          38        0         0 15.00 105.10         13.75
V Kohli  2009 Royal Challengers Bangalore       13   246    219    11     22      8          50        1         0 22.36 112.33         18.92
V Kohli  2010 Royal Challengers Bangalore       13   307    212    11     26     12          58        1         0 27.91 144.81         23.62
V Kohli  2011 Royal Challengers Bangalore       16   557    460    12     55     16          71        4         0 46.42 121.09         34.81
V Kohli  2012 Royal Challengers Bangalore       15   364    326    13     33     10          73        2         0 28.00 111.66         24.27
V Kohli  2013 Royal Challengers Bangalore       16   634    457    14     64     22          99        6         0 45.29 138.73         39.62
V Kohl

In [87]:
df_ball.to_parquet("balls.parquet", index=False)
df_wkts.to_parquet("wickets.parquet", index=False)

In [90]:
import os
tables = {}
for name, f in files.items():
    path = f"{f}.parquet"
    if os.path.exists(path):
        tables[name] = pd.read_parquet(path)
    else:
        print("MISSING:", path)

for name, df in tables.items():
    print(f"\n===== {name}  {df.shape} =====")
    display(df.head(3))


===== df_ball  (278034, 30) =====


,match_id,innings,batting_team,bowling_team,over,ball_in_over,batter,batter_id,bowler,bowler_id,non_striker,runs_batter,runs_extras,runs_total,wides,noballs,byes,legbyes,penalty,is_wide,is_noball,is_wicket,is_legal,batter_ball_faced,bowler_runs,phase,year,is_four,is_six,bat_dot
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,1,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0,powerplay,2017,0,0,1
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,2,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0,powerplay,2017,0,0,1
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,3,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,4,0,4,0,0,0,0,0,False,False,False,True,True,4,powerplay,2017,1,0,0



===== df_wkts  (13794, 18) =====


,match_id,innings,batting_team,bowling_team,over,ball_in_over,player_out,player_out_id,kind,bowler,bowler_id,fielders,striker,non_striker,bowler_wicket,counts_as_out,phase,year
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,5,DA Warner,dcce6f09,caught,A Choudhary,18e6906e,Mandeep Singh,DA Warner,S Dhawan,True,True,powerplay,2017
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,10,3,S Dhawan,0a476045,caught,STR Binny,bd17b45f,Sachin Baby,S Dhawan,MC Henriques,True,True,middle,2017
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,15,2,MC Henriques,32198ae0,caught,YS Chahal,57ee1fde,Sachin Baby,MC Henriques,Yuvraj Singh,True,True,death,2017



===== df_match  (1169, 25) =====


,match_id,date,season_raw,venue,city,team1,team2,toss_winner,toss_decision,winner,result,win_by_runs,win_by_wickets,method,player_of_match,event_name,match_number,stage,year,match_seq,season,venue_raw,super_over_winner,final_winner,is_no_result
0,335982,2008-04-18,2007/08,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,Kolkata Knight Riders,None,140.0,NaN,None,BB McCullum,Indian Premier League,1.0,None,2008,1,2008,M Chinnaswamy Stadium,None,Kolkata Knight Riders,False
1,335983,2008-04-19,2007/08,"PCA IS Bindra Stadium, Mohali",Mohali,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,None,33.0,NaN,None,MEK Hussey,Indian Premier League,2.0,None,2008,2,2008,"Punjab Cricket Association Stadium, Mohali",None,Chennai Super Kings,False
2,335984,2008-04-19,2007/08,"Arun Jaitley Stadium, Delhi",Delhi,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,None,NaN,9.0,None,MF Maharoof,Indian Premier League,3.0,None,2008,3,2008,Feroz Shah Kotla,None,Delhi Capitals,False


... (16 outputs left out)


===== vs_team  (4607, 9) =====


,batter,opposition,innings,runs,balls,outs,high_score,avg,sr
0,A Ashish Reddy,Chennai Super Kings,3,45,25,2,36,22.5,180.000000
1,A Ashish Reddy,Delhi Daredevils,3,36,23,3,16,12.0,156.521739
2,A Ashish Reddy,Kings XI Punjab,3,37,23,1,22,37.0,160.869565



===== at_venue  (6847, 9) =====


,batter,venue,innings,runs,balls,outs,high_score,avg,sr
0,A Ashish Reddy,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,2,28,17,1,15,28.0,164.705882
1,A Ashish Reddy,Eden Gardens,1,4,5,1,4,4.0,80.000000
2,A Ashish Reddy,Feroz Shah Kotla,1,16,9,1,16,16.0,177.777778


In [92]:
# ---------- batting: career table with consistency ----------
def batting_career(bat, season):
    d = bat.copy()
    d["dismissed"] = d["outs"] > 0
    d["contrib"] = (d["runs"] >= 20) | ((d["runs"] >= 10) & (d["strike_rate"] >= 150))
    d["fail"] = (d["runs"] < 10) & d["dismissed"]
    d["fifty_plus"] = d["runs"] >= 50
    d["duck"] = (d["runs"] == 0) & d["dismissed"]

    c = d.groupby("batter").agg(
        innings=("runs", "size"), runs=("runs", "sum"), balls=("balls", "sum"),
        outs=("outs", "sum"), high_score=("runs", "max"), median_runs=("runs", "median"),
        contrib_rate=("contrib", "mean"), fail_rate=("fail", "mean"),
        fifty_rate=("fifty_plus", "mean"), duck_rate=("duck", "mean"),
    ).reset_index()
    c["avg"] = c["runs"] / c["outs"].where(c["outs"] > 0)
    c["sr"] = c["runs"] / c["balls"] * 100
    c["consistency"] = 100 * (0.6 * c["contrib_rate"] + 0.4 * (1 - c["fail_rate"]))

    s = season.sort_values("year").groupby("batter").agg(
        first_year=("year", "min"), last_year=("year", "max"), seasons=("year", "nunique"),
        seasons_300=("runs", lambda x: (x >= 300).sum()),
        teams=("team", lambda t: ", ".join(dict.fromkeys(t))),   # in order played
    ).reset_index()
    return c.merge(s, on="batter", how="left")

career_bat = batting_career(bat, season)
career_bat.to_parquet("career_batting.parquet", index=False)

# ---------- bowling: career table with consistency ----------
def bowling_career(bowl, bseason):
    # totals: ALL innings
    t = bowl.groupby("bowler").agg(
        innings=("balls", "size"), balls=("balls", "sum"),
        runs=("runs", "sum"), wickets=("wickets", "sum")).reset_index()
    t["economy"] = t["runs"] / (t["balls"] / 6)

    # rates: only innings of 2+ overs
    d = bowl[bowl["balls"] >= 12].copy()
    d["took_wkt"] = d["wickets"] >= 1
    d["expensive"] = d["economy"] >= 10
    r = d.groupby("bowler").agg(
        spells=("balls", "size"), median_economy=("economy", "median"),
        wkt_rate=("took_wkt", "mean"), exp_rate=("expensive", "mean")).reset_index()

    c = t.merge(r, on="bowler", how="left")
    c["consistency"] = 100 * (0.5 * c["wkt_rate"] + 0.5 * (1 - c["exp_rate"]))

    s = bseason.sort_values("year").groupby("bowler").agg(
        first_year=("year", "min"), last_year=("year", "max"), seasons=("year", "nunique"),
        teams=("team", lambda x: ", ".join(dict.fromkeys(x)))).reset_index()
    return c.merge(s, on="bowler", how="left")

career_bowl = bowling_career(bowl, bseason)
career_bowl.to_parquet("career_bowling.parquet", index=False)

print(career_bowl[career_bowl["bowler"].isin(["YS Chahal", "JJ Bumrah", "SP Narine", "DJ Bravo"])]
      [["bowler", "innings", "wickets", "economy", "consistency"]].round(2).to_string(index=False))

   bowler  innings  wickets  economy  consistency
 DJ Bravo      158      183     8.38        72.52
JJ Bumrah      145      183     7.25        75.69
SP Narine      187      192     6.80        75.67
YS Chahal      172      221     7.96        75.15


In [98]:
df_ball.head()

,match_id,innings,batting_team,bowling_team,over,ball_in_over,batter,batter_id,bowler,bowler_id,non_striker,runs_batter,runs_extras,runs_total,wides,noballs,byes,legbyes,penalty,is_wide,is_noball,is_wicket,is_legal,batter_ball_faced,bowler_runs,phase,year,is_four,is_six,bat_dot
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,1,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0,powerplay,2017,0,0,1
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,2,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0,powerplay,2017,0,0,1
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,3,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,4,0,4,0,0,0,0,0,False,False,False,True,True,4,powerplay,2017,1,0,0
3,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,4,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,0,0,0,0,0,0,0,False,False,False,True,True,0,powerplay,2017,0,0,1
4,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,0,5,DA Warner,dcce6f09,TS Mills,245c97cb,S Dhawan,0,2,2,2,0,0,0,0,True,False,False,False,False,2,powerplay,2017,0,0,0


In [102]:
df_wkts.head()

,match_id,innings,batting_team,bowling_team,over,ball_in_over,player_out,player_out_id,kind,bowler,bowler_id,fielders,striker,non_striker,bowler_wicket,counts_as_out,phase,year
0,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,1,5,DA Warner,dcce6f09,caught,A Choudhary,18e6906e,Mandeep Singh,DA Warner,S Dhawan,True,True,powerplay,2017
1,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,10,3,S Dhawan,0a476045,caught,STR Binny,bd17b45f,Sachin Baby,S Dhawan,MC Henriques,True,True,middle,2017
2,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,15,2,MC Henriques,32198ae0,caught,YS Chahal,57ee1fde,Sachin Baby,MC Henriques,Yuvraj Singh,True,True,death,2017
3,1082591,1,Sunrisers Hyderabad,Royal Challengers Bengaluru,18,4,Yuvraj Singh,1c914163,bowled,TS Mills,245c97cb,,Yuvraj Singh,DJ Hooda,True,True,death,2017
4,1082591,2,Royal Challengers Bengaluru,Sunrisers Hyderabad,5,4,Mandeep Singh,c3a96caf,bowled,Rashid Khan,5f547c8b,,Mandeep Singh,CH Gayle,True,True,powerplay,2017


In [104]:
df_match.head()

,match_id,date,season_raw,venue,city,team1,team2,toss_winner,toss_decision,winner,result,win_by_runs,win_by_wickets,method,player_of_match,event_name,match_number,stage,year,match_seq,season,venue_raw,super_over_winner,final_winner,is_no_result
0,335982,2008-04-18,2007/08,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,Kolkata Knight Riders,None,140.0,NaN,None,BB McCullum,Indian Premier League,1.0,None,2008,1,2008,M Chinnaswamy Stadium,NaN,Kolkata Knight Riders,False
1,335983,2008-04-19,2007/08,"PCA IS Bindra Stadium, Mohali",Mohali,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,None,33.0,NaN,None,MEK Hussey,Indian Premier League,2.0,None,2008,2,2008,"Punjab Cricket Association Stadium, Mohali",NaN,Chennai Super Kings,False
2,335984,2008-04-19,2007/08,"Arun Jaitley Stadium, Delhi",Delhi,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,None,NaN,9.0,None,MF Maharoof,Indian Premier League,3.0,None,2008,3,2008,Feroz Shah Kotla,NaN,Delhi Capitals,False
3,335986,2008-04-20,2007/08,"Eden Gardens, Kolkata",Kolkata,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,None,NaN,5.0,None,DJ Hussey,Indian Premier League,4.0,None,2008,4,2008,Eden Gardens,NaN,Kolkata Knight Riders,False
4,335985,2008-04-20,2007/08,"Wankhede Stadium, Mumbai",Mumbai,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,None,NaN,5.0,None,MV Boucher,Indian Premier League,5.0,None,2008,5,2008,Wankhede Stadium,NaN,Royal Challengers Bengaluru,False


In [106]:
df_players.head()

,match_id,team,player,player_id
0,1082591,Royal Challengers Bengaluru,CH Gayle,db584dad
1,1082591,Royal Challengers Bengaluru,Mandeep Singh,c3a96caf
2,1082591,Royal Challengers Bengaluru,TM Head,12b610c2
3,1082591,Royal Challengers Bengaluru,KM Jadhav,99d63244
4,1082591,Royal Challengers Bengaluru,SR Watson,4329fbb5


In [108]:
bat.head()

,match_id,innings,batting_team,batter,runs,balls,fours,sixes,outs,date,year,venue,match_seq,opposition,batter_id,strike_rate,inning_no,roll_runs,roll_balls,roll_outs,roll_sr,roll_avg
0,548346,1,Deccan Chargers,A Ashish Reddy,10,10,0,1,1,2012-04-29,2012,Wankhede Stadium,286,Mumbai Indians,1c2a64cd,100.0,1,NaN,NaN,NaN,NaN,NaN
1,548352,2,Deccan Chargers,A Ashish Reddy,3,3,0,0,1,2012-05-04,2012,"MA Chidambaram Stadium, Chepauk",292,Chennai Super Kings,1c2a64cd,100.0,2,NaN,NaN,NaN,NaN,NaN
2,548359,2,Deccan Chargers,A Ashish Reddy,8,8,1,0,1,2012-05-08,2012,"Rajiv Gandhi International Stadium, Uppal",299,Kings XI Punjab,1c2a64cd,100.0,3,NaN,NaN,NaN,NaN,NaN
3,548373,2,Deccan Chargers,A Ashish Reddy,10,4,2,0,0,2012-05-18,2012,"Rajiv Gandhi International Stadium, Uppal",314,Rajasthan Royals,1c2a64cd,250.0,4,NaN,NaN,NaN,NaN,NaN
4,548376,1,Deccan Chargers,A Ashish Reddy,4,4,0,0,1,2012-05-20,2012,"Rajiv Gandhi International Stadium, Uppal",317,Royal Challengers Bangalore,1c2a64cd,100.0,5,35.0,29.0,4.0,120.689655,8.75


In [110]:
season.head()

,batter,year,team,innings,runs,balls,outs,fours,sixes,high_score,fifties,hundreds,avg,sr,runs_per_inn
0,A Ashish Reddy,2012,Deccan Chargers,5,35,29,4,3,1,10,0,0,8.750000,120.689655,7.000000
1,A Ashish Reddy,2013,Sunrisers Hyderabad,10,125,89,6,8,5,36,0,0,20.833333,140.449438,12.500000
2,A Ashish Reddy,2015,Sunrisers Hyderabad,5,73,46,3,3,5,22,0,0,24.333333,158.695652,14.600000
3,A Ashish Reddy,2016,Sunrisers Hyderabad,3,47,29,2,2,4,32,0,0,23.500000,162.068966,15.666667
4,A Badoni,2022,Lucknow Super Giants,11,161,130,8,11,7,54,1,0,20.125000,123.846154,14.636364


In [112]:
bowl.head()

,match_id,innings,team,opposition,bowler,balls,runs,dots,wides,noballs,fours,sixes,wickets,date,year,venue,match_seq,bowler_id,economy,inning_no
0,548341,2,Deccan Chargers,Pune Warriors,A Ashish Reddy,24,32,6,0,0,1,2,2,2012-04-26,2012,"MCA Stadium, Pune",281,1c2a64cd,8.000000,1
1,548346,2,Deccan Chargers,Mumbai Indians,A Ashish Reddy,13,11,8,1,0,1,0,1,2012-04-29,2012,"Wankhede Stadium, Mumbai",286,1c2a64cd,5.076923,2
2,548348,2,Deccan Chargers,Pune Warriors,A Ashish Reddy,18,32,3,0,1,0,2,1,2012-05-01,2012,"Barabati Stadium, Cuttack",288,1c2a64cd,10.666667,3
3,548352,1,Deccan Chargers,Chennai Super Kings,A Ashish Reddy,12,16,3,0,1,1,0,1,2012-05-04,2012,"MA Chidambaram Stadium, Chennai",292,1c2a64cd,8.000000,4
4,548356,2,Deccan Chargers,Royal Challengers Bengaluru,A Ashish Reddy,24,36,11,1,0,3,2,1,2012-05-06,2012,"M Chinnaswamy Stadium, Bengaluru",296,1c2a64cd,9.000000,5


In [114]:
bseason.head()

,bowler,year,team,innings,balls,runs,wickets,dots,best_wkts,three_w,five_w,economy,avg,sr
0,A Ashish Reddy,2012,Deccan Chargers,9,163,237,11,56,3,1,0,8.723926,21.545455,14.818182
1,A Ashish Reddy,2013,Sunrisers Hyderabad,5,40,70,3,11,1,0,0,10.500000,23.333333,13.333333
2,A Ashish Reddy,2015,Sunrisers Hyderabad,4,36,50,3,10,1,0,0,8.333333,16.666667,12.000000
3,A Ashish Reddy,2016,Sunrisers Hyderabad,2,23,39,1,6,1,0,0,10.173913,39.000000,23.000000
4,A Badoni,2022,Lucknow Super Giants,2,12,11,2,4,1,0,0,5.500000,5.500000,6.000000


In [116]:
bat_phase.head()

,batter,year,phase,runs,balls,fours,sixes,dots,outs,sr,avg
0,A Ashish Reddy,2012,death,25,21,3,0,8,4,119.047619,6.25
1,A Ashish Reddy,2012,middle,10,8,0,1,4,0,125.000000,NaN
2,A Ashish Reddy,2013,death,114,78,7,5,19,5,146.153846,22.80
3,A Ashish Reddy,2013,middle,6,5,0,0,1,0,120.000000,NaN
4,A Ashish Reddy,2013,powerplay,5,6,1,0,4,1,83.333333,5.00


In [118]:
bowl_phase.head()

,bowler,year,phase,balls,runs,wickets,economy
0,A Ashish Reddy,2012,death,73,123,8,10.109589
1,A Ashish Reddy,2012,middle,84,94,3,6.714286
2,A Ashish Reddy,2012,powerplay,6,20,0,20.000000
3,A Ashish Reddy,2013,death,10,25,1,15.000000
4,A Ashish Reddy,2013,middle,30,45,2,9.000000


In [120]:
h2h.head()

,batter,bowler,runs,balls,dots,fours,sixes,outs,sr
0,A Ashish Reddy,A Nehra,7,8,4,1,0,1,87.500000
1,A Ashish Reddy,AB Dinda,9,7,1,1,0,0,128.571429
2,A Ashish Reddy,AD Mathews,25,12,2,1,2,0,208.333333
3,A Ashish Reddy,AD Russell,4,3,0,0,0,0,133.333333
4,A Ashish Reddy,Anureet Singh,2,2,0,0,0,0,100.000000


In [122]:
vs_team.head()

,batter,opposition,innings,runs,balls,outs,high_score,avg,sr
0,A Ashish Reddy,Chennai Super Kings,3,45,25,2,36,22.5,180.000000
1,A Ashish Reddy,Delhi Daredevils,3,36,23,3,16,12.0,156.521739
2,A Ashish Reddy,Kings XI Punjab,3,37,23,1,22,37.0,160.869565
3,A Ashish Reddy,Kolkata Knight Riders,2,17,13,2,13,8.5,130.769231
4,A Ashish Reddy,Mumbai Indians,2,27,25,2,17,13.5,108.000000


In [124]:
at_venue.head()

,batter,venue,innings,runs,balls,outs,high_score,avg,sr
0,A Ashish Reddy,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,2,28,17,1,15,28.0,164.705882
1,A Ashish Reddy,Eden Gardens,1,4,5,1,4,4.0,80.000000
2,A Ashish Reddy,Feroz Shah Kotla,1,16,9,1,16,16.0,177.777778
3,A Ashish Reddy,M Chinnaswamy Stadium,2,35,22,2,32,17.5,159.090909
4,A Ashish Reddy,"MA Chidambaram Stadium, Chepauk",2,39,19,1,36,39.0,205.263158


In [126]:
career_bat.head()

,batter,innings,runs,balls,outs,high_score,median_runs,contrib_rate,fail_rate,fifty_rate,duck_rate,avg,sr,consistency,first_year,last_year,seasons,seasons_300,teams
0,A Ashish Reddy,23,280,193,15,36,10.0,0.304348,0.304348,0.000000,0.000000,18.666667,145.077720,46.086957,2012,2016,4,0,"Deccan Chargers, Sunrisers Hyderabad"
1,A Badoni,46,963,695,36,74,18.0,0.521739,0.347826,0.130435,0.043478,26.750000,138.561151,57.391304,2022,2025,4,1,Lucknow Super Giants
2,A Chandila,2,4,7,0,4,2.0,0.000000,0.000000,0.000000,0.000000,NaN,57.142857,40.000000,2012,2013,2,0,Rajasthan Royals
3,A Chopra,6,53,71,6,24,6.5,0.166667,0.500000,0.000000,0.000000,8.833333,74.647887,30.000000,2008,2009,2,0,Kolkata Knight Riders
4,A Choudhary,3,25,20,1,15,6.0,0.000000,0.333333,0.000000,0.000000,25.000000,125.000000,26.666667,2017,2017,1,0,Royal Challengers Bangalore


In [128]:
career_bowl.head()

,bowler,innings,balls,runs,wickets,economy,spells,median_economy,wkt_rate,exp_rate,consistency,first_year,last_year,seasons,teams
0,A Ashish Reddy,20,262,396,18,9.068702,14.0,8.250000,0.857143,0.214286,82.142857,2012,2016,4,"Deccan Chargers, Sunrisers Hyderabad"
1,A Badoni,7,35,49,4,8.400000,NaN,NaN,NaN,NaN,NaN,2022,2025,4,Lucknow Super Giants
2,A Chandila,12,234,242,11,6.205128,12.0,5.291667,0.500000,0.166667,66.666667,2012,2013,2,Rajasthan Royals
3,A Choudhary,5,101,144,5,8.554455,5.0,8.000000,0.800000,0.400000,70.000000,2017,2017,1,Royal Challengers Bengaluru
4,A Dananjaya,1,24,47,0,11.750000,1.0,11.750000,0.000000,1.000000,0.000000,2018,2018,1,Mumbai Indians


In [130]:
import pandas as pd

checks = {
    "balls":                 ("phase",             "phase and year columns"),
    "wickets":               ("phase",             "phase and year columns"),
    "matches":               ("super_over_winner", "tie winners"),
    "player_match_batting":  ("roll_sr",           "rolling form"),
    "career_bowling":        ("spells",            "corrected bowling career table"),
}
for f, (col, why) in checks.items():
    cols = pd.read_parquet(f"{f}.parquet").columns
    print(f"{f:25} {'OK     ' if col in cols else 'MISSING'} {why}")

m = pd.read_parquet("matches.parquet")
print("venues:", m["venue"].nunique(), "(expect 36) | teams:", m["team1"].nunique(), "(expect 15)")

balls                     OK      phase and year columns
wickets                   OK      phase and year columns
matches                   OK      tie winners
player_match_batting      OK      rolling form
career_bowling            OK      corrected bowling career table
venues: 36 (expect 36) | teams: 15 (expect 15)


In [134]:
from pathlib import Path

ROOT = Path(".").resolve()        # or Path(r"C:\path\to\your\project")
SKIP = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules", ".venv", "venv"}

def tree(p, depth=0, max_depth=3):
    if depth > max_depth:
        return
    items = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    shown_json = 0
    for it in items:
        if it.name in SKIP:
            continue
        if it.is_file() and it.suffix == ".json":
            shown_json += 1
            if shown_json > 3:          # don't list all 1,169 match files
                continue
        size = f"  ({it.stat().st_size/1e6:.1f} MB)" if it.is_file() and it.stat().st_size > 1e6 else ""
        print("    " * depth + ("📁 " if it.is_dir() else "") + it.name + size)
        if it.is_dir():
            tree(it, depth + 1, max_depth)
    if shown_json > 3:
        print("    " * depth + f"... and {shown_json - 3} more .json files")


print(ROOT)
tree(ROOT)

C:\Users\Lenovo\ML\PORTFOLIO\Project 1\player-consistency-index\notebooks
01_json_to_ball_by_ball.ipynb
balls.parquet  (1.7 MB)
batter_at_venue.parquet
batter_vs_bowler.parquet
batter_vs_team.parquet
batting_by_phase.parquet
bowling_by_phase.parquet
career_batting.parquet
career_bowling.parquet
matches.parquet
player_match_batting.parquet
player_match_bowling.parquet
player_season_batting.parquet
player_season_bowling.parquet
players.parquet
wickets.parquet


In [136]:
from pathlib import Path

ROOT = Path(r"C:\Users\Lenovo\ML\PORTFOLIO")
SKIP = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
        ".venv", "venv", "env", "site-packages", ".idea", ".vscode"}
MAX_DEPTH = 4       # lower to 3 if the output is too long
MAX_PER_EXT = 4     # files of one type shown per folder; the rest are counted

lines = []

def size_str(p):
    b = p.stat().st_size
    return f"  ({b/1e6:.1f} MB)" if b >= 1e6 else ""

def walk(p, depth=0):
    if depth > MAX_DEPTH:
        lines.append("    " * depth + "... (deeper folders not shown)")
        return
    try:
        items = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    except PermissionError:
        return
    shown = {}
    for it in items:
        if it.name in SKIP:
            continue
        if it.is_dir():
            lines.append("    " * depth + f"[{it.name}]")
            walk(it, depth + 1)
        else:
            ext = it.suffix.lower()
            shown[ext] = shown.get(ext, 0) + 1
            if shown[ext] <= MAX_PER_EXT:
                lines.append("    " * depth + it.name + size_str(it))
    for ext, n in shown.items():
        if n > MAX_PER_EXT:
            lines.append("    " * depth + f"... and {n - MAX_PER_EXT} more {ext or 'extensionless'} files")

lines.append(str(ROOT))
walk(ROOT)
out = "\n".join(lines)
print(out)
Path("portfolio_tree.txt").write_text(out, encoding="utf-8")

C:\Users\Lenovo\ML\PORTFOLIO
[IPL json ball by ball]
    ipl_json.zip  (3.9 MB)
[Project 1]
    [player-consistency-index]
        [data]
            [processed]
            [raw]
                [ipl_json]
                    ... (deeper folders not shown)
        [notebooks]
            01_json_to_ball_by_ball.ipynb
            balls.parquet  (1.7 MB)
            batter_at_venue.parquet
            batter_vs_bowler.parquet
            batter_vs_team.parquet
            ... and 11 more .parquet files
[Project 2]
    [data]
        [processed]
            matches.csv
        [raw]
            [ipl_json]
                1082591.json
                1082592.json
                1082593.json
                1082594.json
                README.txt
                ... and 1165 more .json files
    [notebooks]
        match_outcome_analysis.ipynb
    .gitignore
    requirements.txt


888